     -------------------------------------- 126.3/126.3 KB 2.5 MB/s eta 0:00:00
     ---------------------------------------- 63.8/63.8 KB ? eta 0:00:00
  Using cached click-8.1.8-py3-none-any.whl (98 kB)
     ---------------------------------------- 75.8/75.8 KB 4.1 MB/s eta 0:00:00
     ---------------------------------------- 6.0/6.0 MB 19.2 MB/s eta 0:00:00
     --------------------------------------- 21.7/21.7 MB 22.6 MB/s eta 0:00:00
  Using cached hyperlink-21.0.0-py2.py3-none-any.whl (74 kB)
  Using cached rich-14.2.0-py3-none-any.whl (243 kB)
     ------------------------------------- 506.1/506.1 KB 16.0 MB/s eta 0:00:00
  Using cached shellingham-1.5.4-py2.py3-none-any.whl (9.8 kB)
  Using cached tomli-2.3.0-py3-none-any.whl (14 kB)
  Using cached jaraco.classes-3.4.0-py3-none-any.whl (6.8 kB)
  Using cached jaraco_functools-4.3.0-py3-none-any.whl (10 kB)
  Using cached jaraco.context-6.0.1-py3-none-any.whl (6.8 kB)
  Using cached pywin32_ctypes-0.2.3-py3-none-any.whl (30 kB)

You should consider upgrading via the 'd:\claimpkg\claimpkg-clone\.venv\Scripts\python.exe -m pip install --upgrade pip' command.
UsageError: Line magic function `%hatch` not found.


In [48]:
RESULT_FILE = r'D:\claimpkg\claimpkg-clone\src\relabelling\result.json'
DATASET_PICKLE_FILE = r'D:\claimpkg\claimpkg-clone\src\resources\classified_book_dataset_6300.pickle'
RELABEL_QUEUE_FILE = r'D:\claimpkg\claimpkg-clone\src\resources\relabel_queue.pickle'

import pickle
import re
import sys
sys.path.append('..')
from utils.parser import tuple_to_str

# Open json file but read as text
with open(RESULT_FILE, 'r', encoding='utf-8') as f:
    json_text = f.read()

dataset = pickle.load(open(DATASET_PICKLE_FILE, 'rb'))
relabel_queue = pickle.load(open(RELABEL_QUEUE_FILE, 'rb'))

In [49]:
def str_to_triplet(s: str) -> list[tuple]:
    triplet_strings = s.split(';\n')

    for s in triplet_strings:
        triplet = str_object_to_triplet(s)
    return [str_object_to_triplet(s) for s in triplet_strings]

def str_object_to_triplet(s: str) -> tuple[str, str, str]:
    match = re.match(r'<e>(.*?)</e>\s*\|\|\s*(.*?)\s*\|\|\s*<e>(.*?)</e>', s.strip())
    if match:
        entity1, relation, entity2 = match.groups()
        return (entity1, relation, entity2)
    else:
        raise ValueError(f"String does not match triplet format: {s}")

# Split the string into individual labelled objects where each object starts with {"claim": and ends with "created_by": <integer>}
labelled_objects = re.findall(r'(\{"claim":.*?"created_by": "\d+"\})', json_text, re.DOTALL)
# Turn these strings into dictionaries
import json
labelled_dicts = [json.loads(obj) for obj in labelled_objects]
# # Test str_to_triplet function on the 'triplets' field of the first labelled object
# INDEX = 29
# print(f"Claim: {labelled_dicts[INDEX]['claim']}")
# print(f"Strings: {labelled_dicts[INDEX]['triplets']}")
# str_to_triplet(labelled_dicts[INDEX]['triplets'])
formal_dict = {}
for labelled in labelled_dicts:
    try:
        claim = labelled['claim']
        triplets = str_to_triplet(labelled['triplets'])
        formal_dict[claim] = triplets
    except ValueError as e:
        print(f"Error processing claim: {labelled['claim']}")
        print(e)
print(f"Total labelled claims: {len(formal_dict)}")

# Export formal_dict to a pickle file
with open(r'D:\claimpkg\claimpkg-clone\src\resources\formal_labelled_dict.pickle', 'wb') as f:
    pickle.dump(formal_dict, f)

Error processing claim: The Atlantic City International Airport is located at the Four World Trade Centre.
String does not match triplet format: 
Error processing claim: Her real name is Michelle Suárez Bértora.
String does not match triplet format: 
Total labelled claims: 1375


In [50]:
old_dataset

{'is published by Lippincott Williams & Wilkins in the UK where English is the main language.': {'Label': [True],
  'Entity_set': ['United_Kingdom',
   'English_language',
   'Lippincott_Williams_&_Wilkins',
   'AIDS_(journal)'],
  'Evidence': {'United_Kingdom': [['language'], ['~country']],
   'AIDS_(journal)': [['country'], ['publisher']],
   'English_language': [['~language']],
   'Lippincott_Williams_&_Wilkins': [['~publisher']]},
  'types': ['coll:model', 'num4', 'multi claim'],
  'triplet': '<e>United Kingdom</e> || language || <e>English language</e>\n<e>AIDS (journal)</e> || country || <e>United Kingdom</e>\n<e>AIDS (journal)</e> || publisher || <e>Lippincott Williams & Wilkins</e>',
  'is_concern': True,
  'complexity': 'easy'},
 'Cliff Louis plays for the Virginia Destroyers team.': {'Label': [False],
  'Entity_set': ['Cliff_Louis', 'Virginia_Destroyers'],
  'Evidence': {'Cliff_Louis': [['team']], 'Virginia_Destroyers': [['~team']]},
  'types': ['coll:model', 'num1', 'substit

In [51]:
# Merging formal_dict into dataset with classified_book_dataset_6300.pickle
CLASSIFIED_DATASET_FILE = r'D:\claimpkg\claimpkg-clone\src\resources\classified_book_dataset_6300.pickle'
new_dataset = {}
old_dataset = pickle.load(open(CLASSIFIED_DATASET_FILE, 'rb'))
# Adding items from old_dataset to new_dataset where it's complexity: 'easy'
for claim, data in old_dataset.items():
    if data['complexity'] == 'easy' or data['complexity'] == 'skipped':
        new_dataset[claim] = data
# Finding keys in formal_dict
formal_dict_keys = set(formal_dict.keys())
# Adding items from old_dataset where keys are in formal_dict_keys

changed_counts = 0
for key in formal_dict_keys:
    if key in old_dataset:
        new_dataset[key] = old_dataset[key]
        new_dataset[key]['triplet'] = formal_dict[key]
        changed_counts += 1
    else:
        print(f"Key {key} not found in old_dataset")
print(f"Total changed counts: {changed_counts}")
print(f"Total new dataset counts: {len(new_dataset)}")

Total changed counts: 1375
Total new dataset counts: 5924


In [52]:
# Suffle the new_dataset
import random
new_dataset_items = list(new_dataset.items())
random.shuffle(new_dataset_items)
new_dataset = dict(new_dataset_items)
keys = list(new_dataset.keys())
errors = []
for i, key in enumerate(keys):
    triplet_str = []
    for triplet in new_dataset[key]['triplet']:
        try:
            triplet_str.append(f"<e>{triplet[0]}</e> || {triplet[1]} || <e>{triplet[2]}</e>")
        except Exception as e:
            errors.append(key)
    new_dataset[key]['triplet'] = "\n".join(triplet_str)
# Remove error keys
old_count = len(new_dataset)
for key in errors:
    del new_dataset[key]
new_count = len(new_dataset)
print(f"Removed {old_count - new_count} entries due to errors.")

# Split train : 4500, test: 1000, val: other
train_dataset = {}
test_dataset = {}
val_dataset = {}
for i, key in enumerate(keys):
    if i < 4500:
        train_dataset[key] = new_dataset[key]
    elif i < 5500:
        test_dataset[key] = new_dataset[key]
    else:
        val_dataset[key] = new_dataset[key]

# Export
DIR = r'D:\claimpkg\claimpkg-clone\src\resources'
with open(f'{DIR}/factkg_train_4500_triplets.pickle', 'wb') as f:
    pickle.dump(train_dataset, f)
with open(f'{DIR}/factkg_test_1000_triplets.pickle', 'wb') as f:
    pickle.dump(test_dataset, f)
with open(f'{DIR}/factkg_val_424_triplets.pickle', 'wb') as f:
    pickle.dump(val_dataset, f)

# Try to load those exported files
with open(f'{DIR}/factkg_train_4500_triplets.pickle', 'rb') as f:
    loaded_train = pickle.load(f)
with open(f'{DIR}/factkg_test_1000_triplets.pickle', 'rb') as f:
    loaded_test = pickle.load(f)
with open(f'{DIR}/factkg_val_424_triplets.pickle', 'rb') as f:
    loaded_val = pickle.load(f)
print(f"Loaded train dataset size: {len(loaded_train)}")
print(f"Loaded test dataset size: {len(loaded_test)}")
print(f"Loaded val dataset size: {len(loaded_val)}")

# print("Sample entries from new_dataset:")
# for i in range(5):
#     key = keys[i]
#     print(f"Claim: {key}")
#     print(f"Triplet: {new_dataset[key]['triplet']}")
#     print("-----")

Removed 0 entries due to errors.
Loaded train dataset size: 4500
Loaded test dataset size: 1000
Loaded val dataset size: 424
